# Kaggle Housing Price Prediction - Final Ensemble

Combines predictions from multiple independently-trained models into a weighted ensemble. Each model predicts `log(SalePrice)` and the weights are optimised by minimizing RMSE on out-of-fold (OOF) predictions. The final submission is converted back to raw price.

**Models:**
- XGBoost (from `bdt-model.ipynb`)
- LightGBM (from `bdt-model.ipynb`)
- Huber (from `linear-model.ipynb`)

**Pipeline:**
1. Load saved OOF (train) and test predictions from each model
2. Optimise ensemble weights with constraint that weights sum to 1
3. Generate weighted ensemble prediction in log-space
4. Convert to raw price and write submission CSV

In [6]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

## Load Predictions

Each model notebook saves two `.npy` files:
- **OOF predictions** on the training set (generated via `cross_val_predict` so each sample is predicted by a model that never saw it)
- **Test predictions** on the held-out Kaggle test set

All predictions are in **log-space** (`log(SalePrice)`).

In [7]:
# OOF predictions (log-space) for weight optimisation
xgb_oof   = np.load('./output/ensemble-input-predictions/xgb_oof.npy')
lgbm_oof  = np.load('./output/ensemble-input-predictions/lgbm_oof.npy')
huber_oof = np.load('./output/ensemble-input-predictions/huber_oof.npy')

# test predictions (log-space) for final prediction
xgb_test   = np.load('./output/ensemble-input-predictions/xgb_test.npy')
lgbm_test  = np.load('./output/ensemble-input-predictions/lgbm_test.npy')
huber_test = np.load('./output/ensemble-input-predictions/huber_test.npy')

# true training targets (log-space)
train = pd.read_csv('./input/train.csv.gz', compression='gzip', index_col='Id')
test  = pd.read_csv('./input/test.csv.gz', compression='gzip', index_col='Id')

# the OOF arrays correspond to the training rows AFTER outlier removal,
# so we need the same filtered target here
mask = ~((train['Neighborhood'] == 'Edwards') &
         (train['SaleCondition'] == 'Partial') &
         (train['GrLivArea'] > 4000))
log_y_train = np.log(train.loc[mask, 'SalePrice'].values)

print(f'OOF shapes:  XGB {xgb_oof.shape}, LGBM {lgbm_oof.shape}, Huber {huber_oof.shape}')
print(f'Test shapes: XGB {xgb_test.shape}, LGBM {lgbm_test.shape}, Huber {huber_test.shape}')
print(f'Train target: {log_y_train.shape}')

OOF shapes:  XGB (1458,), LGBM (1458,), Huber (1458,)
Test shapes: XGB (1459,), LGBM (1459,), Huber (1459,)
Train target: (1458,)


## Optimise Ensemble Weights

Find the weight vector $\vec{w} = (w_\mathrm{XGB},\; w_\mathrm{LGBM},\; w_\mathrm{Huber})$ that minimizes RMSE on OOF predictions, subject to the constraint that all weights sum to $1$ and are positive:

$$\sum w_i = 1 ~~\mathrm{and}~~ w_i \geq 0$$

In [8]:
oof_preds   = [xgb_oof, lgbm_oof, huber_oof]
test_preds  = [xgb_test, lgbm_test, huber_test]
model_names = ['XGB', 'LGBM', 'Huber']


def ensemble_rmse(weights):
    '''
    RMSE of the weighted ensemble on OOF predictions
    '''
    blended = sum(w * oof for w, oof in zip(weights, oof_preds))
    return np.sqrt(np.mean((blended - log_y_train) ** 2))


result = minimize(ensemble_rmse,
                  x0=np.ones(len(oof_preds)) / len(oof_preds),
                  bounds=[(0, 1)] * len(oof_preds),
                  constraints={'type': 'eq', 'fun': lambda w: sum(w) - 1},
                  method='SLSQP',
                  )

weights = result.x

print('Optimal weights:')
for name, w in zip(model_names, weights):
    print(f'  {name:>8s}: {w:.4f}')

print(f'\nIndividual model RMSE:')
for name, oof in zip(model_names, oof_preds):
    rmse = np.sqrt(np.mean((oof - log_y_train) ** 2))
    print(f'  {name:>8s}: {rmse:.5f}')

print(f'\nEnsemble RMSE: {ensemble_rmse(weights):.5f}')

Optimal weights:
       XGB: 0.5696
      LGBM: 0.0000
     Huber: 0.4304

Individual model RMSE:
       XGB: 0.11158
      LGBM: 0.11575
     Huber: 0.11450

Ensemble RMSE: 0.10761


As we saw in the BDT model already, the XGB and LGBM predictions are very correlated because both tree models are quite similar. When the weights were optimized for the BDT model, LGBM already had an optimal weight of ~$0.15$. With the addition of the linear model, which is significantly different from the tree-based models, the marginal contribution of LGBM is likely quite minimal so it is reasonable that it adds no benefit to the full ensemble.

The linear model makes different mistakes than the tree models so combining them averages out the error which is where the performance gain comes from.

## Generate Submission

Apply the optimised weights to the test predictions (still in log-space), then exponentiate to get raw `SalePrice`.

In [9]:
log_ensemble_preds = sum(w * pred for w, pred in zip(weights, test_preds))
ensemble_preds     = np.exp(log_ensemble_preds)

output = pd.DataFrame({'Id': test.index, 'SalePrice': ensemble_preds})
output.to_csv('./output/ensemble-submission.csv', index=False)

print(f'Submission saved with {output.shape[0]} rows')
output.head()

Submission saved with 1459 rows


,Id,SalePrice
0,1461,121217.915711
1,1462,161526.579875
2,1463,185999.348657
3,1464,193701.031116
4,1465,192287.676272
